In [1]:
import os
import uuid
import chromadb

from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader

In [2]:
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="document_collection",
    embedding_function=embedding_function
)

print("ChromaDB collection ready")

ChromaDB collection ready


In [4]:
try:
    client.delete_collection("document_collection")
except:
    pass

collection = client.get_or_create_collection(
    name="document_collection",
    embedding_function=embedding_function
)

print("Collection reset complete")

Collection reset complete


In [5]:
documents = []
metadatas = []

pdf_folder = "data/pdfs"
txt_folder = "data/txt"

# Load PDFs
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        docs = loader.load()

        for d in docs:
            documents.append(d.page_content)
            metadatas.append({
                "source": file,
                "page": d.metadata.get("page")
            })

# Load TXT files
for file in os.listdir(txt_folder):
    if file.endswith(".txt"):
        loader = TextLoader(os.path.join(txt_folder, file))
        docs = loader.load()

        for d in docs:
            documents.append(d.page_content)
            metadatas.append({
                "source": file
            })

print("Documents loaded:", len(documents))

Documents loaded: 5


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunk_texts = []
chunk_metadatas = []

for doc, meta in zip(documents, metadatas):

    chunks = text_splitter.split_text(doc)

    for chunk in chunks:
        chunk_texts.append(chunk)
        chunk_metadatas.append(meta)

print("Chunks created:", len(chunk_texts))
print("Metadata entries:", len(chunk_metadatas))

Chunks created: 188
Metadata entries: 188


In [7]:
collection.add(
    ids=[str(uuid.uuid4()) for _ in chunk_texts],
    documents=chunk_texts,
    metadatas=chunk_metadatas
)

print("Chunks stored in ChromaDB")

Chunks stored in ChromaDB


In [8]:
query = input("Enter your question: ")
k = int(input("Enter number of nearest answers to retrieve: "))

In [9]:
results = collection.query(
    query_texts=[query],
    n_results=k
)

print("\nQuery:", query)
print("\nRelevant Results:\n")


Query: what is internal audit

Relevant Results:



In [10]:
for i, doc in enumerate(results["documents"][0]):

    metadata = results["metadatas"][0][i]

    source = metadata.get("source", "Unknown")
    page = metadata.get("page", "N/A")

    print(f"Result {i+1}")
    print("Source:", source)
    print("Page:", page)
    print(doc)
    print("--------------------------------------------------")

Result 1
Source: Internal-Audit-Framework-1.pdf
Page: 2
for the function being audited and who have the authority to take action on the internal audit 
recommendations. Internal audit reports are confidential documents and their distribution should 
therefore be restricted to those managers who need to know, the Audit Committee and the External  
Auditors. 
 
The Internal Auditor shall produce clear, constructive and concise written reports based on sufficient,  
relevant and reliable evidence, which: 
 
- state the scope, purpose, extent and conclusions of the internal audit assignment 
- acknowledge the actions taken, or proposed, by management 
 
The Internal Auditor shall also prepare flash reports to alert management on the need to take control, or 
when there are reasonable grounds for suspicion of any major irregularity fraud or theft.
--------------------------------------------------
